# Extração dos arquivos da PGE

In [1]:
import pdfplumber
import pandas as pd

def extract_pge_optimized(pdf_path):
    all_rows = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # A estratégia 'lines' é mais precisa para tabelas com grades
            table = page.extract_table({
                "vertical_strategy": "lines",
                "horizontal_strategy": "lines",
                "intersection_y_tolerance": 10
            })
            
            if table:
                all_rows.extend(table)

    # 1. Criar DataFrame inicial e remover lixo de cabeçalho
    df = pd.DataFrame(all_rows)
    
    # Localiza onde a tabela real começa (ignora metadados do topo)
    # Procuramos pela linha que contém "Contribuinte" ou "N° Origem"
    header_idx = df[df.apply(lambda x: x.astype(str).str.contains('Contribuinte').any(), axis=1)].index
    if not header_idx.empty:
        df = df.iloc[header_idx[0] + 1:].reset_index(drop=True)

    # 2. Definir nomes das colunas conforme o padrão do documento
    df.columns = [
        "Contribuinte", "N_Origem", "Nosso_Numero", "Pagamento", 
        "Principal", "Multa", "Juros", "SubTotal", 
        "Convenio_BB", "Honorarios", "Pago"
    ]

    # 3. TRATAMENTO DE LINHAS QUEBRADAS (O Pulo do Gato)
    # Se 'Pago' estiver vazio, a linha provavelmente é continuação da anterior
    df['is_continuation'] = df['Pago'].apply(lambda x: x is None or str(x).strip() == "")
    
    final_rows = []
    current_row = None

    for _, row in df.iterrows():
        # Se a linha tem valor em 'Pago', é o início (ou fim) de um registro completo [cite: 413, 526]
        if not row['is_continuation']:
            if current_row is not None:
                final_rows.append(current_row)
            current_row = row.tolist()
        else:
            # Se for continuação, concatena o texto nas colunas de Identificação [cite: 392, 501]
            if current_row:
                current_row[0] = f"{current_row[0]} {row['Contribuinte']}".strip()
                current_row[1] = f"{current_row[1]} {row['N_Origem']}".strip()

    # Adiciona a última linha processada
    if current_row:
        final_rows.append(current_row)

    # 4. Criar o DataFrame final limpo
    df_final = pd.DataFrame(final_rows, columns=df.columns).drop(columns=['is_continuation'])

    # 5. Limpeza de dados numéricos [cite: 413, 633]
    cols_to_fix = ["Principal", "Multa", "Juros", "SubTotal", "Convenio_BB", "Honorarios", "Pago"]
    for col in cols_to_fix:
        df_final[col] = df_final[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

    # Remove linhas de subtotal/total que restaram do rodapé [cite: 409, 629]
    df_final = df_final.dropna(subset=['Pago'])
    
    return df_final

# Uso
# df_jan = extract_pge_optimized("Anexo_38723414_TCE_jan_2024.pdf")

In [2]:
import os
from pathlib import Path

output_dir = Path("saidas/pge")

pdf_dir = r"C:\Users\05911205424\Documents\arquivos_pge"

dfs = []

for pdf_file in os.listdir(pdf_dir):
    if pdf_file.endswith(".pdf"):
        x = pdf_file
        stem = Path(x).stem
        ano, mes = stem.split("_")[-2:]
        new_file_name = output_dir / f"planilha_pge_{ano}_{mes}.xlsx"
        pdf_path = os.path.join(pdf_dir, pdf_file)
        df = extract_pge_optimized(pdf_path)
        #df.to_excel(os.path.join(pdf_dir, new_file_name), index=False)
        dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
#df.to_excel(output_dir / "planilha_pge_completa.xlsx", index=False)


In [3]:
import re

processos = list(df['N_Origem'].apply(lambda x: [item.strip().lower().replace("-tc", "") for item in x.split(";")]))

In [4]:
processos_flat = [
    proc.strip()
    for sublist in processos
    for proc in (sublist if isinstance(sublist, list) else [sublist])
    if str(proc).strip()
]

processos_flat = [
    p for p in processos_flat
    if isinstance(p, str) and re.fullmatch(r"\d+/\d+", p.strip())
]

processos_flat = [
    f"{p.split('/', 1)[0].zfill(6)}/{p.split('/', 1)[1]}"
    for p in processos_flat
]


In [5]:
processos = list(set(processos_flat))

In [6]:
processos_enviados = ['000061/2025',
'000064/2025',
'000066/2025',
'000067/2025',
'000068/2025',
'000081/2021',
'000081/2022',
'000081/2023',
'000082/2021',
'000082/2022',
'000083/2021',
'000083/2023',
'000084/2021',
'000084/2022',
'000084/2023',
'000085/2021',
'000085/2022',
'000086/2021',
'000086/2022',
'000086/2023',
'000087/2021',
'000087/2022',
'000087/2023',
'000088/2022',
'000089/2021',
'000089/2023',
'000090/2023',
'000091/2022',
'000091/2023',
'000092/2021',
'000092/2022',
'000093/2021',
'000093/2022',
'000094/2021',
'000095/2021',
'000095/2022',
'000095/2023',
'000096/2021',
'000097/2021',
'000098/2021',
'000100/2021',
'000101/2023',
'000102/2021',
'000102/2023',
'000103/2021',
'000104/2021',
'000104/2023',
'000105/2021',
'000107/2021',
'000109/2021',
'000110/2021',
'000112/2021',
'000112/2023',
'000113/2021',
'000114/2021',
'000116/2021',
'000120/2021',
'000122/2021',
'000122/2023',
'000123/2021',
'000123/2023',
'000124/2021',
'000124/2023',
'000126/2023',
'000127/2023',
'000129/2021',
'000132/2021',
'000133/2021',
'000134/2021',
'000136/2023',
'000137/2021',
'000138/2023',
'000139/2023',
'000140/2021',
'000140/2023',
'000141/2021',
'000142/2021',
'000143/2023',
'000144/2023',
'000145/2021',
'000146/2021',
'000147/2021',
'000148/2021',
'000149/2021',
'000149/2023',
'000150/2021',
'000151/2021',
'000152/2021',
'000153/2021',
'000153/2022',
'000153/2023',
'000154/2021',
'000154/2022',
'000154/2023',
'000155/2021',
'000155/2022',
'000157/2021',
'000158/2021',
'000158/2023',
'000160/2021',
'000160/2022',
'000160/2023',
'000321/2020',
'000322/2020',
'000323/2020',
'000324/2020',
'000327/2020',
'000330/2020',
'000331/2020',
'000332/2020',
'000333/2020',
'000335/2020',
'000338/2020',
'000339/2020',
'000341/2020',
'000342/2020',
'000343/2020',
'000344/2020',
'000345/2020',
'000346/2020',
'000369/2020',
'000370/2020',
'000371/2020',
'000372/2020',
'000373/2020',
'000375/2020',
'000376/2020',
'000378/2020',
'000384/2020',
'000385/2020',
'000388/2020',
'000389/2020',
'000390/2020',
'000391/2020',
'000393/2020',
'000394/2020',
'000399/2020',
'000401/2025',
'000402/2024',
'000403/2024',
'000404/2025',
'000405/2024',
'000406/2024',
'000406/2025',
'000407/2024',
'000408/2024',
'000410/2025',
'000412/2024',
'000413/2024',
'000413/2025',
'000447/2024',
'000463/2024',
'000465/2024',
'000471/2024',
'000474/2024',
'000475/2024',
'000479/2024',
'000724/2025',
'000729/2025',
'000732/2025',
'000763/2019',
'000765/2019',
'000768/2019',
'000770/2019',
'000773/2019',
'000774/2019',
'000775/2019',
'000776/2019',
'000789/2019',
'000794/2015',
'000796/2019',
'000797/2019',
'000801/2019',
'000806/2019',
'000808/2019',
'000814/2019',
'000815/2019',
'000825/2019',
'000828/2019',
'000830/2019',
'000831/2019',
'000835/2019',
'000837/2019',
'000851/2025',
'000852/2025',
'000857/2025',
'000858/2025',
'000861/2025',
'000865/2025',
'000866/2025',
'000875/2025',
'000878/2025',
'000881/2017',
'000886/2017',
'000891/2017',
'000899/2017',
'000913/2017',
'000914/2017',
'000924/2016',
'000952/2017',
'000954/2017',
'001364/2021',
'001365/2021',
'001367/2022',
'001369/2022',
'001370/2021',
'001371/2021',
'001372/2021',
'001373/2021',
'001374/2021',
'001374/2022',
'001375/2021',
'001375/2022',
'001376/2021',
'001376/2022',
'001377/2021',
'001377/2022',
'001378/2022',
'001379/2021',
'001380/2022',
'001381/2022',
'001382/2021',
'001382/2022',
'001383/2021',
'001384/2021',
'001385/2021',
'001386/2021',
'001387/2021',
'001387/2022',
'001387/2023',
'001388/2022',
'001389/2021',
'001390/2022',
'001391/2021',
'001391/2022',
'001392/2021',
'001392/2022',
'001392/2023',
'001393/2021',
'001393/2022',
'001394/2021',
'001394/2022',
'001394/2023',
'001395/2021',
'001395/2022',
'001395/2023',
'001396/2021',
'001397/2021',
'001397/2022',
'001397/2023',
'001398/2021',
'001398/2022',
'001398/2023',
'001399/2021',
'001399/2022',
'001399/2023',
'001400/2021',
'001400/2022',
'001401/2021',
'001401/2022',
'001401/2023',
'001402/2021',
'001402/2022',
'001402/2023',
'001403/2021',
'001403/2022',
'001403/2023',
'001404/2021',
'001404/2022',
'001404/2023',
'001405/2021',
'001405/2022',
'001406/2022',
'001407/2021',
'001407/2022',
'001407/2023',
'001408/2022',
'001408/2023',
'001409/2021',
'001409/2022',
'001409/2023',
'001411/2023',
'001412/2021',
'001412/2023',
'001413/2021',
'001413/2022',
'001413/2023',
'001414/2021',
'001414/2023',
'001415/2021',
'001415/2022',
'001415/2023',
'001416/2022',
'001418/2021',
'001418/2022',
'001419/2021',
'001419/2022',
'001420/2021',
'001420/2022',
'001421/2022',
'001422/2021',
'001422/2022',
'001423/2021',
'001424/2021',
'001424/2023',
'001425/2021',
'001426/2021',
'001426/2022',
'001427/2021',
'001427/2022',
'001427/2023',
'001428/2021',
'001428/2022',
'001428/2023',
'001429/2021',
'001429/2022',
'001430/2022',
'001430/2023',
'001431/2023',
'001432/2021',
'001432/2022',
'001432/2023',
'001433/2022',
'001433/2023',
'001435/2021',
'001435/2023',
'001436/2023',
'001437/2021',
'001437/2023',
'001438/2023',
'001439/2021',
'001439/2023',
'001440/2021',
'001440/2023',
'001442/2023',
'001448/2023',
'001449/2023',
'001451/2023',
'001453/2023',
'001455/2023',
'001458/2023',
'001461/2023',
'001483/2025',
'001485/2025',
'001489/2025',
'001490/2025',
'001495/2025',
'001496/2025',
'001497/2025',
'001842/2025',
'001850/2025',
'001861/2018',
'001862/2018',
'001863/2025',
'001872/2018',
'001873/2018',
'001876/2018',
'001884/2018',
'001889/2018',
'001895/2018',
'001904/2018',
'001915/2018',
'002002/2024',
'002011/2024',
'002011/2025',
'002012/2024',
'002012/2025',
'002014/2024',
'002014/2025',
'002015/2025',
'002016/2025',
'002017/2025',
'002031/2024',
'002033/2024',
'002035/2024',
'002059/2024',
'002063/2024',
'002125/2019',
'002128/2019',
'002132/2019',
'002133/2019',
'002136/2019',
'002137/2019',
'002141/2019',
'002146/2019',
'002147/2019',
'002151/2019',
'002152/2019',
'002152/2025',
'002153/2025',
'002155/2019',
'002156/2019',
'002160/2019',
'002161/2019',
'002162/2019',
'002163/2019',
'002166/2019',
'002168/2025',
'002169/2025',
'002170/2019',
'002170/2025',
'002171/2019',
'002172/2019',
'002174/2019',
'002175/2019',
'002176/2019',
'002180/2019',
'002184/2019',
'002185/2019',
'002187/2019',
'002189/2019',
'002194/2019',
'002391/2025',
'002395/2025',
'002397/2025',
'002399/2025',
'002405/2025',
'002407/2025',
'002408/2025',
'002409/2025',
'002541/2024',
'002561/2018',
'002566/2020',
'002569/2018',
'002569/2020',
'002569/2024',
'002570/2018',
'002572/2018',
'002572/2020',
'002573/2018',
'002573/2020',
'002573/2024',
'002574/2018',
'002574/2020',
'002575/2020',
'002577/2024',
'002578/2018',
'002578/2020',
'002579/2020',
'002580/2020',
'002581/2018',
'002581/2020',
'002582/2020',
'002583/2024',
'002585/2020',
'002586/2020',
'002587/2020',
'002588/2020',
'002589/2020',
'002590/2020',
'002591/2020',
'002591/2024',
'002593/2020',
'002593/2024',
'002594/2020',
'002594/2024',
'002595/2020',
'002598/2020',
'002598/2024',
'002599/2020',
'002600/2020',
'002601/2020',
'002603/2020',
'002608/2020',
'002609/2024',
'002609/2025',
'002610/2020',
'002610/2024',
'002611/2020',
'002611/2025',
'002612/2020',
'002612/2024',
'002613/2020',
'002613/2025',
'002614/2020',
'002614/2024',
'002615/2020',
'002616/2020',
'002616/2024',
'002617/2020',
'002617/2024',
'002618/2020',
'002618/2024',
'002619/2020',
'002620/2020',
'002620/2025',
'002621/2020',
'002623/2020',
'002625/2018',
'002626/2020',
'002628/2018',
'002629/2018',
'002629/2020',
'002630/2020',
'002632/2020',
'002633/2020',
'002686/2022',
'002687/2022',
'002688/2022',
'002690/2022',
'002691/2022',
'002692/2022',
'002693/2022',
'002694/2022',
'002697/2022',
'002699/2022',
'002700/2022',
'002703/2022',
'002705/2022',
'002707/2022',
'002709/2022',
'002713/2022',
'002714/2022',
'002719/2022',
'002724/2025',
'002726/2025',
'002727/2025',
'002730/2022',
'002731/2022',
'002733/2025',
'002734/2025',
'002735/2022',
'002736/2025',
'002740/2022',
'002744/2022',
'002749/2022',
'002750/2022',
'002751/2022',
'002754/2022',
'002755/2022',
'002757/2022',
'002758/2022',
'002812/2025',
'002813/2025',
'002816/2025',
'002817/2025',
'002818/2025',
'002822/2025',
'002961/2021',
'002962/2021',
'002963/2021',
'002965/2021',
'002967/2021',
'002968/2021',
'002969/2021',
'002971/2021',
'002972/2021',
'002973/2021',
'002974/2021',
'002975/2021',
'002976/2021',
'002977/2021',
'002979/2021',
'002980/2021',
'002981/2021',
'002982/2021',
'002983/2021',
'002984/2021',
'002986/2021',
'002987/2021',
'002989/2021',
'002990/2021',
'002991/2021',
'002992/2021',
'002993/2021',
'002994/2021',
'002997/2021',
'002999/2021',
'003001/2021',
'003003/2021',
'003004/2021',
'003005/2021',
'003005/2022',
'003006/2021',
'003007/2021',
'003008/2021',
'003009/2021',
'003011/2021',
'003012/2022',
'003013/2022',
'003014/2021',
'003014/2022',
'003015/2021',
'003015/2022',
'003016/2022',
'003017/2021',
'003017/2022',
'003018/2021',
'003018/2022',
'003019/2021',
'003019/2022',
'003021/2021',
'003022/2022',
'003023/2021',
'003024/2021',
'003024/2022',
'003025/2022',
'003026/2021',
'003027/2021',
'003027/2022',
'003028/2021',
'003028/2022',
'003029/2021',
'003029/2022',
'003030/2021',
'003030/2022',
'003031/2022',
'003032/2021',
'003032/2022',
'003033/2021',
'003033/2022',
'003034/2021',
'003034/2022',
'003035/2021',
'003035/2022',
'003036/2021',
'003037/2021',
'003037/2022',
'003038/2022',
'003039/2022',
'003040/2025',
'003041/2022',
'003043/2022',
'003043/2025',
'003044/2022',
'003045/2022',
'003046/2022',
'003047/2025',
'003049/2025',
'003050/2022',
'003051/2022',
'003059/2022',
'003060/2022',
'003062/2022',
'003265/2023',
'003268/2023',
'003273/2023',
'003275/2023',
'003277/2023',
'003281/2023',
'003283/2023',
'003289/2023',
'003294/2023',
'003297/2023',
'003302/2023',
'003304/2023',
'003305/2023',
'003306/2023',
'003308/2023',
'003345/2024',
'003348/2024',
'003349/2024',
'003351/2024',
'003352/2024',
'003353/2024',
'003354/2024',
'003355/2024',
'003358/2024',
'003360/2024',
'003364/2024',
'003365/2024',
'003367/2024',
'003369/2024',
'003372/2024',
'003375/2024',
'003376/2024',
'003377/2024',
'003378/2024',
'003386/2024',
'003387/2024',
'003389/2024',
'003390/2024',
'003393/2024',
'003395/2024',
'003400/2024',
'003402/2024',
'003404/2024',
'003406/2024',
'003407/2024',
'003416/2024',
'003417/2024',
'003418/2024',
'003419/2024',
'003481/2019',
'003483/2019',
'003488/2019',
'003489/2019',
'003490/2019',
'003491/2019',
'003496/2019',
'003498/2019',
'003500/2019',
'003505/2019',
'003507/2019',
'003508/2019',
'003509/2019',
'003512/2019',
'003513/2019',
'003514/2019',
'003516/2019',
'003517/2019',
'003520/2019',
'003522/2019',
'003523/2019',
'003524/2019',
'003525/2019',
'003526/2019',
'003527/2019',
'003528/2019',
'003530/2019',
'003532/2019',
'003534/2019',
'003535/2019',
'003536/2019',
'003537/2019',
'003539/2019',
'003540/2019',
'003542/2019',
'003543/2019',
'003548/2019',
'003549/2019',
'003552/2019',
'003554/2019',
'003555/2019',
'003556/2019',
'003557/2019',
'003558/2019',
'003560/2019',
'003621/2017',
'003641/2022',
'003644/2022',
'003645/2022',
'003650/2022',
'003651/2022',
'003652/2022',
'003655/2022',
'003656/2022',
'003658/2022',
'003660/2022',
'003662/2022',
'003663/2025',
'003664/2025',
'003665/2022',
'003666/2025',
'003669/2025',
'003670/2025',
'003671/2025',
'003673/2025',
'003674/2025',
'003682/2022',
'003683/2022',
'003684/2022',
'003685/2022',
'003688/2022',
'003689/2022',
'003698/2022',
'003702/2022',
'003703/2022',
'003705/2022',
'003706/2022',
'003710/2022',
'003712/2022',
'003714/2022',
'003715/2022',
'003718/2022',
'003719/2022',
'004081/2018',
'004086/2018',
'004103/2018',
'004104/2018',
'004109/2018',
'004110/2018',
'004111/2018',
'004114/2018',
'004115/2018',
'004117/2018',
'004119/2018',
'004124/2018',
'004137/2018',
'004153/2018',
'004158/2018',
'004321/2021',
'004322/2021',
'004323/2021',
'004324/2021',
'004441/2022',
'004443/2022',
'004445/2022',
'004446/2022',
'004447/2022',
'004771/2024',
'004865/2024',
'004866/2024',
'004868/2024',
'004874/2024',
'004875/2024',
'004876/2024',
'004877/2024',
'004880/2024',
'004882/2024',
'004884/2024',
'004885/2024',
'004886/2024',
'004891/2024',
'004893/2024',
'004894/2024',
'004897/2024',
'004899/2024',
'004900/2024',
'004909/2024',
'004910/2024',
'004913/2024',
'004918/2024',
'005247/2019',
'005248/2019',
'005250/2019',
'005251/2019',
'005252/2019',
'005255/2019',
'005259/2019',
'005260/2019',
'005261/2019',
'005263/2019',
'005264/2019',
'005266/2019',
'005267/2019',
'005269/2019',
'005270/2019',
'005271/2019',
'005275/2019',
'005277/2019',
'005279/2019',
'005280/2019',
'005282/2017',
'005282/2019',
'005283/2019',
'005284/2019',
'005287/2019',
'005288/2019',
'005289/2019',
'005290/2019',
'005291/2019',
'005292/2019',
'005293/2019',
'005295/2017',
'005295/2019',
'005296/2019',
'005298/2019',
'005300/2019',
'005303/2019',
'005304/2019',
'005305/2019',
'005307/2017',
'005308/2019',
'005310/2019',
'005313/2019',
'005315/2019',
'005316/2017',
'005316/2019',
'005318/2019',
'005319/2019',
'005321/2019',
'005322/2019',
'005323/2017',
'005335/2017',
'005346/2017',
'005451/2016',
'005607/2018',
'005619/2018',
'005620/2018',
'005625/2018',
'005629/2018',
'005634/2018',
'005638/2018',
'005641/2018',
'005644/2018',
'005648/2018',
'005649/2018',
'005658/2018',
'005665/2018',
'005673/2018',
'005674/2018',
'005676/2018',
'005677/2018',
'006325/2018',
'006326/2018',
'006334/2018',
'006335/2018',
'006340/2018',
'006346/2018',
'006347/2018',
'006348/2018',
'006352/2018',
'006363/2018',
'006366/2018',
'006371/2018',
'006373/2018',
'006375/2018',
'006376/2018',
'006377/2018',
'006383/2018',
'006387/2018',
'006388/2018',
'006389/2018',
'006390/2018',
'006391/2018',
'006397/2018',
'007406/2019',
'007410/2019',
'007412/2019',
'007414/2019',
'007415/2019',
'007416/2019',
'007420/2019',
'007422/2019',
'007423/2019',
'007425/2019',
'007427/2019',
'007428/2019',
'007429/2019',
'007430/2019',
'007432/2019',
'007434/2019',
'007435/2019',
'007436/2019',
'007437/2019',
'007438/2019',
'007439/2019',
'007440/2019',
'007441/2019',
'007442/2019',
'007443/2019',
'007445/2019',
'007446/2019',
'007448/2019',
'007449/2019',
'007450/2019',
'007451/2019',
'007453/2019',
'007455/2019',
'007475/2013',
'009145/2013',
'009200/2018',
'009201/2018',
'009203/2018',
'009204/2018',
'009205/2017',
'009206/2018',
'009210/2018',
'009215/2018',
'009217/2018',
'009219/2018',
'009220/2018',
'009221/2017',
'009261/2018',
'009263/2018',
'009264/2018',
'009265/2018',
'009266/2018',
'009269/2018',
'009270/2018',
'009271/2018',
'009272/2018',
'009273/2018',
'010860/2016',
'011024/2016',
'011040/2016',
'011314/2018',
'011317/2018',
'011318/2018',
'011323/2018',
'011330/2018',
'011335/2018',
'011337/2018',
'011338/2018',
'011342/2018',
'011346/2018',
'011347/2018',
'011349/2018',
'011356/2018',
'011360/2018',
'011365/2018',
'011369/2018',
'011370/2018',
'011371/2018',
'011375/2018',
'011376/2018',
'011377/2018',
'011378/2018',
'011379/2018',
'011847/2017',
'011862/2017',
'011865/2017',
'011889/2017',
'011894/2017',
'011898/2017',
'011909/2017',
'011916/2017',
'012966/2017',
'012979/2017',
'012993/2017',
'013013/2017',
'013025/2017',
'013038/2017',
'013040/2017',
'015281/2017',
'015282/2017',
'015300/2017',
'015302/2017',
'015310/2017',
'015311/2017',
'015327/2017',
'015342/2017',
'015353/2017',
'015354/2017',
'015388/2015',
'015759/2016',
'016739/2016',
'016743/2016',
'016746/2016',
'016753/2016',
'017975/2016',
'017979/2016',
'018013/2016',
'018014/2016',
'018016/2016',
'018018/2016',
'018022/2016',
'018023/2016',
'018024/2016',
'018026/2016',
'018040/2016',
'018727/2015',
'019436/2017',
'019439/2017',
'019441/2017',
'019445/2017',
'019448/2017',
'019452/2017',
'019454/2017',
'019457/2017',
'019462/2017',
'021691/2016',
'021700/2016']

In [7]:
processos_fora_sicode = sorted(set(processos) - set(processos_enviados))

In [8]:
processos_fora_sicode

['000408/2003',
 '000544/2011',
 '000594/2004',
 '000638/2003',
 '000821/2007',
 '000936/2004',
 '001150/2010',
 '001158/1997',
 '001332/2011',
 '001333/2011',
 '001336/2011',
 '001513/2008',
 '001693/2008',
 '001694/2008',
 '001695/2008',
 '001705/2008',
 '001707/2008',
 '001713/2008',
 '001714/2008',
 '001722/2008',
 '001751/2008',
 '001757/2008',
 '001760/2008',
 '001771/2008',
 '001773/2008',
 '001788/2008',
 '001799/2002',
 '001805/2008',
 '001821/2008',
 '001825/2008',
 '001826/2008',
 '001827/2008',
 '001847/2011',
 '001945/2008',
 '002280/2008',
 '002296/2008',
 '002340/1999',
 '002371/1999',
 '002444/2005',
 '002552/2003',
 '002709/2008',
 '002717/2008',
 '003032/1998',
 '003425/2003',
 '003597/2008',
 '003659/2003',
 '003719/2003',
 '003764/2010',
 '003790/2009',
 '003806/2009',
 '003836/2009',
 '003852/2003',
 '003863/2004',
 '003871/2008',
 '003888/2007',
 '003894/2007',
 '003925/2003',
 '003926/2007',
 '003939/2007',
 '004014/2007',
 '004021/2007',
 '004033/2008',
 '004063

In [9]:
set(processos) - set(processos_fora_sicode)

{'000066/2025',
 '000084/2021',
 '000086/2022',
 '000086/2023',
 '000087/2022',
 '000089/2023',
 '000091/2023',
 '000092/2021',
 '000094/2021',
 '000097/2021',
 '000098/2021',
 '000102/2023',
 '000113/2021',
 '000116/2021',
 '000120/2021',
 '000124/2021',
 '000132/2021',
 '000139/2023',
 '000140/2021',
 '000141/2021',
 '000144/2023',
 '000146/2021',
 '000149/2021',
 '000153/2021',
 '000153/2022',
 '000154/2021',
 '000155/2022',
 '000158/2021',
 '000323/2020',
 '000344/2020',
 '000370/2020',
 '000375/2020',
 '000403/2024',
 '000732/2025',
 '000773/2019',
 '000857/2025',
 '000875/2025',
 '000914/2017',
 '001365/2021',
 '001367/2022',
 '001377/2022',
 '001378/2022',
 '001381/2022',
 '001387/2021',
 '001388/2022',
 '001392/2023',
 '001393/2021',
 '001395/2021',
 '001395/2022',
 '001396/2021',
 '001397/2022',
 '001398/2022',
 '001400/2021',
 '001402/2022',
 '001402/2023',
 '001403/2022',
 '001404/2023',
 '001405/2022',
 '001407/2022',
 '001408/2023',
 '001409/2023',
 '001413/2022',
 '001414

In [10]:
df['processos'] = df['N_Origem'].apply(lambda x: [item.strip().lower().replace("-tc", "") for item in x.split(";")])
df['processos'] = df['processos'].apply(lambda x: [f"{item.split('/', 1)[0].zfill(6)}/{item.split('/', 1)[1]}" for item in x if re.fullmatch(r"\d+/\d+", item.strip())])

In [11]:
def is_in_sicode(processos):
    return any(proc in processos_enviados for proc in processos)

df['no_sicode'] = df['processos'].apply(is_in_sicode)

In [12]:
df = df[df["processos"].apply(lambda x: isinstance(x, list) and len(x) >0)]


In [13]:
df_fora_sicode = df[~df['no_sicode']]

In [14]:
df_fora_sicode

,Contribuinte,N_Origem,Nosso_Numero,Pagamento,Principal,Multa,Juros,SubTotal,Convenio_BB,Honorarios,Pago,processos,no_sicode
0,FRANCISCO ANASTACIO DE ASSIS:\n229.913.104-30,5917/2010,000965621,26/01/2024,35.60,0.00,52.73,88.33,0.0,3.57,91.90,[005917/2010],False
1,CLAUDIO CHRISTIAN BEZERRIL DA\nSILVA: 388.370....,001945/2008-tc,000962961,23/01/2024,54.48,0.00,40.33,94.81,0.0,3.59,98.40,[001945/2008],False
3,ROBERTO LUIZ: 307.333.754-72,005182/2006-TC,000964438,24/01/2024,43.43,5.53,31.19,80.15,0.0,3.59,83.74,[005182/2006],False
4,ROBERTO LUIZ: 307.333.754-72,005182/2006-TC,000964440,24/01/2024,43.43,2.72,31.19,77.34,0.0,3.59,80.93,[005182/2006],False
5,IDALINA NETA DE LIMA VIANA:\n074.352.154-49,011448/2003-TC,000967205,31/01/2024,51.67,0.00,25.58,77.25,0.0,3.60,80.85,[011448/2003],False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3682,GILSON GERALDO DE OLIVEIRA:\n406.691.814-04,700545/2010-TC,001270100,31/12/2025,474.57,0.00,490.31,964.88,0.0,81.48,1046.36,[700545/2010],False
3683,MARIA JOSE OLIVEIRA:\n874.984.804-68,013198/2010-tc,001254838,01/12/2025,483.03,0.00,671.38,1154.41,0.0,88.63,1243.04,[013198/2010],False
3684,MARIA JOSE OLIVEIRA:\n874.984.804-68,013198/2010-tc,001268607,31/12/2025,483.03,0.00,681.61,1164.64,0.0,88.63,1253.27,[013198/2010],False
3686,ELISIO BRITO DE MEDEIROS\nGALVAO: 444.234.204-06,009716/2010,001268745,26/12/2025,618.15,62.91,1050.55,1731.61,0.0,110.04,1841.65,[009716/2010],False


In [17]:
df_fora_sicode['processo'] = df_fora_sicode['processos'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else None)

C:\Users\05911205424\AppData\Local\Temp\ipykernel_22060\1606169224.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fora_sicode['processo'] = df_fora_sicode['processos'].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else None)


In [60]:
tab_1 = df_fora_sicode.groupby(['processo'])['Pago'].sum().reset_index()

In [62]:
tab_2 = df_fora_sicode

In [63]:
arquivo_saida = output_dir / "processos_fora_sicode.xlsx"

with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
    tab_1.to_excel(writer, sheet_name="total", index=False)
    tab_2.to_excel(writer, sheet_name="processos", index=False)

print(f"Arquivo salvo em: {arquivo_saida}")

Arquivo salvo em: saidas\pge\processos_fora_sicode.xlsx


# Checar débitos

In [18]:
df_fora_sicode

,Contribuinte,N_Origem,Nosso_Numero,Pagamento,Principal,Multa,Juros,SubTotal,Convenio_BB,Honorarios,Pago,processos,no_sicode,processo
0,FRANCISCO ANASTACIO DE ASSIS:\n229.913.104-30,5917/2010,000965621,26/01/2024,35.60,0.00,52.73,88.33,0.0,3.57,91.90,[005917/2010],False,005917/2010
1,CLAUDIO CHRISTIAN BEZERRIL DA\nSILVA: 388.370....,001945/2008-tc,000962961,23/01/2024,54.48,0.00,40.33,94.81,0.0,3.59,98.40,[001945/2008],False,001945/2008
3,ROBERTO LUIZ: 307.333.754-72,005182/2006-TC,000964438,24/01/2024,43.43,5.53,31.19,80.15,0.0,3.59,83.74,[005182/2006],False,005182/2006
4,ROBERTO LUIZ: 307.333.754-72,005182/2006-TC,000964440,24/01/2024,43.43,2.72,31.19,77.34,0.0,3.59,80.93,[005182/2006],False,005182/2006
5,IDALINA NETA DE LIMA VIANA:\n074.352.154-49,011448/2003-TC,000967205,31/01/2024,51.67,0.00,25.58,77.25,0.0,3.60,80.85,[011448/2003],False,011448/2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3682,GILSON GERALDO DE OLIVEIRA:\n406.691.814-04,700545/2010-TC,001270100,31/12/2025,474.57,0.00,490.31,964.88,0.0,81.48,1046.36,[700545/2010],False,700545/2010
3683,MARIA JOSE OLIVEIRA:\n874.984.804-68,013198/2010-tc,001254838,01/12/2025,483.03,0.00,671.38,1154.41,0.0,88.63,1243.04,[013198/2010],False,013198/2010
3684,MARIA JOSE OLIVEIRA:\n874.984.804-68,013198/2010-tc,001268607,31/12/2025,483.03,0.00,681.61,1164.64,0.0,88.63,1253.27,[013198/2010],False,013198/2010
3686,ELISIO BRITO DE MEDEIROS\nGALVAO: 444.234.204-06,009716/2010,001268745,26/12/2025,618.15,62.91,1050.55,1731.61,0.0,110.04,1841.65,[009716/2010],False,009716/2010


In [21]:
import sys
parent_dir = Path.cwd().resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

from utils_ccd import get_connection

In [23]:
processos_fora_sicode = df_fora_sicode['processo'].dropna().unique().tolist()
sql_pge = f''' 
SELECT deb.Status_PGE, *
FROM processo.dbo.Exe_Debito deb
INNER JOIN processo.dbo.Processos exe ON deb.IdProcessoExecucao = exe.IdProcesso
WHERE deb.Status_PGE IS NOT NULL
AND CONCAT(exe.numero_processo, '/', exe.ano_processo) IN ({','.join([f"'{proc}'" for proc in processos_fora_sicode])})
'''
df_pge = pd.read_sql(sql_pge, con=get_connection())


In [24]:
df_pge

,Status_PGE,IdDebito,IdDebitoAnterior,IdProcessoExecucao,IdProcessoOrigem,IdInformacaoDecisao,IdInformacaoTransito,dataDecisao,dataAto,dataTransito,...,ObservacaoDIN,IdProcessoApensador,TipoDiligencia,DataUltimoRecebimento,TramitacaoAberta,IdModalidade,UsuarioProtocoloEletronico,TipoSustentacaoOral,idPublicidadeProcesso,hipotese
